# Spring vector stroke study v0.1

Frozen F2 C (70/30), three independent black-vector methods. No composition changes, color or G-code. Set `DATASCAPE_DATA_DIR` to the directory holding the original TIFFs. Run in the installed project environment: `pip install -e ".[research,dev]"`. All final previews read the same exported stroke records.

Here, **composition C** always means the frozen 70/30 scene. **Stroke C** is only the direction-field method; it does not select a new composition.

In [ ]:
from pathlib import Path
import os, json
from IPython.display import display, Image
from datascape.rendering.vector_study import run
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
data = Path(os.environ.get("DATASCAPE_DATA_DIR", root / "data/raw/seoul"))
output = Path(os.environ.get("DATASCAPE_VECTOR_OUTPUT", root / "outputs/spring_vector_study_v01"))
config = root / "configs/scenes/seoul_spring_vector_study_v01.json"
print(json.loads(config.read_text()))

## Generate and inspect

A uses transverse terrain rows and a checked chord-length spline. B uses signed-curvature axes and slope-supported elevation curves. C integrates the projected aspect/Hessian tangent field. Every method shares the selected omission, nominal 0.32 mm opaque line width, seed and archived vegetation geometry. Candidate counts are not normalized; measured densities are reported.

In [ ]:
record = run(data / "SeoulNorth_v02_MASTER_30m.tif", data / "SeoulNorth_v02_VEG_10m.tif", config, output, root)
display(Image(filename=str(output / "ABC_comparison.png")))
display(Image(filename=str(output / "physical_comparison.png")))

## Inspect actual path removal

Gray edge pixels in these previews are antialiasing, not stroke opacity. No white paint or renderer-only omission mask is used. JSON polylines are finalized at the panel seam; SVG uses their exact coordinates. The source-derived marks remain schematic, not observed trees.

In [ ]:
from datascape.strokes.vector import read_vectors
assert all(record["invariants"].values())
for method, metrics in record["metrics"].items():
    payload = read_vectors(output / f"{method}_strokes.json.gz")
    assert all(s["opacity"] == 1 for s in payload["strokes"])
    assert metrics["checks"]["zero_core_sample_violations"] == 0
    assert metrics["checks"]["last_2mm_footprint_empty"]
    after = metrics["after_omission"]
    print(method, after["terrain_vectors"], round(after["terrain_length_mm"] / 1000, 2), "m", metrics["length_retained_fraction"])
display(Image(filename=str(output / "depth_details.png")))
display(Image(filename=str(output / "omission_comparison.png")))